In [1]:
import runBBN
import matplotlib.pyplot as plt
import ODESolve_slow as ODE
from constants import mN
import numpy as np
import thermodynamics as thermo
from constants import me, mpl, zeta3, hbar, MeVtoT9, cmgstoMeV
import weakrates as weak
import expansion as ex
import nseabundance as nse
import derivatives as der
import xy_reactions as xy
import xgamma_reactions as xg

In [2]:
def sep(z):
    return z[0], z[1], z[2], z[3:]
T,t,eta,A = sep(np.arange(12))

#creates array of 12 dependent variables
def depvar(T,t,eta,A):
    return np.concatenate((np.array([T,t,eta]), A))

In [3]:
def a_total_y_tot(eta_final):
    
    a0 = 0.1    
    T0 = 10
    t0 = 0
    eta0 = eta_final*(11/4)
    Yp0 = (1+(weak.Npntot(T0,a0))/(weak.Nnptot(T0,a0)) )**(-1)
    Yn0 = 1-Yp0

    y0 = depvar(T0, t0, eta0, nse.nse(T0, eta0, Yp0, Yn0))
    p = 0
    dx0=0.001

    N_step = 1000
    dN = 1

    x_final = 0.4

    a, y, dx, done = ODE.ODEOneRun(a0, y0, dx0, p, N_step, dN, x_final)
    print(done)

    a1 = a[-1]
    T1 = y[-1, 0]
    t1 = y[-1, 1]
    eta1 = y[-1, 2]
    Y1 = der.depvar(T1,t1,eta1,nse.nse(T1,eta1,y[-1,3], y[-1,4]))
    dx1 = dx[-1]*0.001
    p = 2
    Nstep = 1000
    dN = 20
    x_final = 100

    a2, y2, dx2, done = ODE.ODEOneRun(a1, Y1, dx1, p, Nstep, dN, x_final)
    print(done)
    for i in range(len(a)):
        A_nse = nse.nse(y[i,0], y[i,2], y[i,3],y[i,4])
        y[i,3:] = A_nse   

    a_total = np.concatenate((a[:-1],a2))   #creates an array of all the ind var values
    #len(a_total) = len(a)-1+len(a2)
    y_total = np.concatenate((y[:-1,:],y2),axis = 0) #creates a matrix of dep var values
    #y_total.shape is len(a_total) x 12

    return a_total, y_total

In [4]:
%%time
a_total, y_total  = a_total_y_tot(16.2e-9)

True
True
CPU times: user 22.7 s, sys: 568 ms, total: 23.3 s
Wall time: 23 s


In [5]:
print(y_total[-1,2])

6.131989717974598e-10


In [9]:
np.savez("Q6", a_total = a_total, y_total = y_total)